In [47]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster, playercareerstats

In [48]:
lakers = teams.find_teams_by_full_name('los angeles lakers')
lakers_id = lakers[0]['id']
celtics = teams.find_teams_by_full_name('boston celtics')
celtics_id = celtics[0]['id']
knicks = teams.find_teams_by_full_name('new york knicks')
knicks_id = knicks[0]['id']

In [49]:
lakers_roster = commonteamroster.CommonTeamRoster(team_id=lakers_id).get_data_frames()[0]
celtics_roster = commonteamroster.CommonTeamRoster(team_id=celtics_id).get_data_frames()[0]
knicks_roster = commonteamroster.CommonTeamRoster(team_id=knicks_id).get_data_frames()[0]

In [93]:
import pandas as pd
import json

def transform_to_dynamo_json(df, details):
    # Create a list of Maps. Each map has 'name' (String) and 'player_id' (Number)
    player_list = [
        {
            "M": {
                "name": {"S": row['PLAYER']},
                "player_id": {"N": str(row['PLAYER_ID'])}
            }
        } 
        for _, row in df.iterrows()
    ]
    
    print(f"Processed {len(player_list)} players for {details['full_name']}")

    dynamo_item = {
        "team_id": {
            "N": str(df['TeamID'].iloc[0]) # Using iloc for cleaner access
        },
        "name": {
            "S": details['full_name']
        },
        "team": {
            "M": {
                "players": {
                    "L": player_list
                }
            }
        }
    }
    
    return dynamo_item

# Your existing logic to save the files
teams_to_process = [
    (lakers_roster, lakers[0], 'lakers'),
    (celtics_roster, celtics[0], 'celtics'),
    (knicks_roster, knicks[0], 'knicks')
]

for roster_df, team_details, slug in teams_to_process:
    formatted_data = transform_to_dynamo_json(roster_df, team_details)
    with open(f'player_data/formatted_data_{slug}.json', 'w') as f:
        json.dump(formatted_data, f, indent=4)

Processed 18 players for Los Angeles Lakers
Processed 15 players for Boston Celtics
Processed 18 players for New York Knicks


In [94]:
import os
import json

player_data_dir = 'player_data'
# Filter specifically for the 'formatted_data_' files we just created
player_files = [f for f in os.listdir(player_data_dir) if f.startswith('formatted_data_') and f.endswith('.json')]

all_player_data = {}

for file in player_files:
    with open(os.path.join(player_data_dir, file), 'r') as f:
        data = json.load(f)
        
        # Extract the ID and the player list
        t_id = data['team_id']['N']
        
        # UPDATED PROCESSING: 
        # We now dig into the Map 'M' to get the 'player_id'
        players_in_file = [
            player['M']['player_id']['N'] 
            for player in data['team']['M']['players']['L']
        ]
        
        # Store in our dictionary
        if t_id in all_player_data:
            all_player_data[t_id].extend(players_in_file)
        else:
            all_player_data[t_id] = players_in_file

# Save the combined mapping for your next script
with open('combined_team_player_map.json', 'w') as f:
    json.dump(all_player_data, f, indent=4)

print(f"Combined data from {len(player_files)} teams. Total teams in map: {len(all_player_data)}")

Combined data from 3 teams. Total teams in map: 3


In [106]:
import time
import json
import os
from tqdm import tqdm
from nba_api.stats.endpoints import playercareerstats, commonplayerinfo

TABLE_NAME = 'Players'
batch_items = []
batch_count = 0

for team_id, player_ids in all_player_data.items():
    for player_id in tqdm(player_ids, desc=f"Processing Team {team_id}"):
        try:
            # 1. Fetch Career Stats (Shooting Data)
            career = playercareerstats.PlayerCareerStats(player_id=player_id)
            career_df = career.get_data_frames()[0]

            # 2. Fetch Player Bio Info (Biographical Data)
            player_info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
            info_df = player_info.get_data_frames()[0]

            if not career_df.empty and not info_df.empty:
                stats_row = career_df.iloc[-1] 
                bio_row = info_df.iloc[0]

                # Shooting Ingredients
                p3_1 = int(stats_row['FG3M'])
                p3_0 = int(stats_row['FG3A'] - stats_row['FG3M'])
                p2_1 = int(stats_row['FGM'] - stats_row['FG3M'])
                p2_0 = int((stats_row['FGA'] - stats_row['FG3A']) - (stats_row['FGM'] - stats_row['FG3M']))
                ft1 = int(stats_row['FTM'])
                ft0 = int(stats_row['FTA'] - stats_row['FTM'])

                # Constructing the Headshot URL
                headshot_url = f"https://cdn.nba.com/headshots/nba/latest/1040x760/{player_id}.png"

                # Constructing the Item with Bio, Headshot, and Nested Stats
                put_request = {
                    "PutRequest": {
                        "Item": {
                            "player_id": {"N": str(player_id)},
                            "team_id": {"N": str(team_id)},
                            "name": {"S": str(bio_row['DISPLAY_FIRST_LAST'])},
                            "headshot_url": {"S": headshot_url}, # NEW ATTRIBUTE
                            "birthdate": {"S": str(bio_row['BIRTHDATE'])},
                            "height": {"S": str(bio_row['HEIGHT'])},
                            "weight": {"S": str(bio_row['WEIGHT'])},
                            "jersey": {"S": str(bio_row['JERSEY'])},
                            "position": {"S": str(bio_row['POSITION'])},
                            "draft_year": {"S": str(bio_row['DRAFT_YEAR'])},
                            "stats": {
                                "M": {
                                    "p2_1": {"N": str(p2_1)},
                                    "p2_0": {"N": str(p2_0)},
                                    "p3_1": {"N": str(p3_1)},
                                    "p3_0": {"N": str(p3_0)},
                                    "ft1": {"N": str(ft1)},
                                    "ft0": {"N": str(ft0)},
                                    "total_points": {"N": str(stats_row['PTS'])}
                                }
                            }
                        }
                    }
                }
                
                batch_items.append(put_request)

            # Batching logic: Save every 25 items
            if len(batch_items) == 25:
                output_path = f'player_data/batch_{batch_count}.json'
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                with open(output_path, 'w') as f:
                    json.dump({TABLE_NAME: batch_items}, f, indent=4)
                
                batch_items = []
                batch_count += 1

            time.sleep(0.8) 
        except Exception as e:
            print(f" Error for {player_id}: {e}")
            continue

# Final catch for remaining items
if batch_items:
    output_path = f'player_data/batch_{batch_count}.json'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump({TABLE_NAME: batch_items}, f, indent=4)

print(f"\nDone! Created batch files with headshot URLs.")

Processing Team 1610612752: 100%|██████████| 18/18 [00:22<00:00,  1.26s/it]


Done! Created batch files with headshot URLs.


In [107]:
from nba_api.stats.endpoints import commonplayerinfo
wiggs = commonplayerinfo.CommonPlayerInfo(203952)
wiggs.get_data_frames()[0]

# render the image from the url
url = "https://cdn.nba.com/headshots/nba/latest/1040x760/2544.png"

# render the image from the url
from IPython.display import Image
# Image(url)

In [28]:
from nba_api.stats.endpoints.scoreboardv3 import ScoreboardV3
scoreboard = ScoreboardV3(game_date="2026-05-03")
scoreboard.game_header.get_data_frame()

,gameId,gameCode,gameStatus,gameStatusText,period,gameClock,gameTimeUTC,gameEt,regulationPeriods,seriesGameNumber,gameLabel,gameSubLabel,seriesText,ifNecessary,seriesConference,poRoundDesc,gameSubtype,isNeutral
0,0042500107,20260503/ORLDET,3,Final,4,,2026-05-03T19:30:00Z,2026-05-03T15:30:00Z,4,Game 7,East First Round,Game 7,DET wins 4-3,False,East,1st Round,,False
1,0042500137,20260503/TORCLE,3,Final,4,,2026-05-03T23:30:00Z,2026-05-03T19:30:00Z,4,Game 7,East First Round,Game 7,CLE wins 4-3,False,East,1st Round,,False


In [45]:
from nba_api.stats.endpoints.playerdashboardbylastngames import PlayerDashboardByLastNGames
pdbng = PlayerDashboardByLastNGames(player_id=203952, last_n_games=10)
pdbng.get_data_frames()[0].columns

Index(['GROUP_SET', 'GROUP_VALUE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM',
       'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT',
       'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD',
       'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3',
       'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK',
       'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK',
       'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK',
       'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK',
       'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK',
       'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK',
       'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT'],
      dtype='object')

## Team player rank (export)

Uses `PlayerDashboardByLastNGames` Last-10 totals per roster player, then **within each team** sums z-scores for PTS, REB, AST, STL, BLK and subtracts z-score for TOV. Writes `player_rank_export.json` in this folder (same directory as `roster.ipynb` when cwd is `backend/`).

In [50]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from nba_api.stats.endpoints.playerdashboardbylastngames import PlayerDashboardByLastNGames
from nba_api.stats.library.parameters import Season, SeasonType

def fetch_last10_row(player_id, season, season_type):
    try:
        ep = PlayerDashboardByLastNGames(
            player_id=int(player_id),
            last_n_games=10,
            season=season,
            season_type_playoffs=season_type,
            timeout=30,
        )
        df = ep.last10_player_dashboard.get_data_frame()
        if df is None or df.empty:
            return None
        return df.iloc[0].to_dict()
    except Exception:
        return None

def team_rank_composite(rows):
    if not rows:
        return []
    df = pd.DataFrame(rows).reset_index(drop=True)
    pos = ["PTS", "REB", "AST", "STL", "BLK"]
    neg = ["TOV"]
    zsum = pd.Series(0.0, index=df.index, dtype=float)
    for c in pos:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
        std = float(s.std(ddof=0))
        m = float(s.mean())
        z = (s - m) / std if std > 1e-9 else pd.Series(0.0, index=df.index)
        zsum = zsum + z
    for c in neg:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
        std = float(s.std(ddof=0))
        m = float(s.mean())
        z = (s - m) / std if std > 1e-9 else pd.Series(0.0, index=df.index)
        zsum = zsum - z
    out = []
    for i in df.index:
        d = df.iloc[int(i)].to_dict()
        d["composite_z"] = float(zsum.iloc[int(i)])
        out.append(d)
    out.sort(key=lambda x: x["composite_z"], reverse=True)
    for idx, d in enumerate(out, start=1):
        d["team_rank"] = idx
    return out

season = Season.default
season_type = SeasonType.regular
roster_specs = [
    # (lakers_roster, int(lakers_id), lakers[0].get("full_name", "Lakers")),
    # (celtics_roster, int(celtics_id), celtics[0].get("full_name", "Celtics")),
    (knicks_roster, int(knicks_id), knicks[0].get("full_name", "Knicks")),
]
teams_out = []
for roster_df, tid, tname in roster_specs:
    base_rows = []
    for _, row in roster_df.iterrows():
        pid = row.get("PLAYER_ID")
        pname = row.get("PLAYER")
        if pid is None or (isinstance(pid, float) and pd.isna(pid)):
            continue
        pid = int(pid)
        snap = fetch_last10_row(pid, season, season_type)
        time.sleep(0.55)
        if snap is None:
            base_rows.append({"PLAYER_ID": pid, "PLAYER": str(pname), "fetch_ok": False})
            continue
        rec = dict(snap)
        rec["PLAYER_ID"] = pid
        rec["PLAYER"] = str(pname)
        rec["fetch_ok"] = True
        base_rows.append(rec)
    ok_rows = [r for r in base_rows if r.get("fetch_ok") and "PTS" in r]
    ranked = team_rank_composite(ok_rows)
    id_to_rank = {int(r["PLAYER_ID"]): r for r in ranked if r.get("PLAYER_ID") is not None}
    players_export = []
    for r in base_rows:
        pid = int(r["PLAYER_ID"])
        last10 = {k: v for k, v in r.items() if k not in ("PLAYER_ID", "PLAYER", "fetch_ok")}
        entry = {"player_id": pid, "name": r["PLAYER"], "fetch_ok": r.get("fetch_ok", False), "last10": last10}
        if pid in id_to_rank:
            entry["composite_z"] = id_to_rank[pid]["composite_z"]
            entry["team_rank"] = id_to_rank[pid]["team_rank"]
        else:
            entry["composite_z"] = None
            entry["team_rank"] = None
        players_export.append(entry)
    teams_out.append({"team_id": tid, "team_name": tname, "players": players_export})

here = Path.cwd()
out_path = (here / "player_rank_export.json") if (here / "server.js").exists() else (here / "backend" / "player_rank_export.json")

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "season": season,
    "season_type": season_type,
    "last_n_games": 10,
    "endpoint": "playerdashboardbylastngames",
    "dataset": "Last10PlayerDashboard",
    "rank_method": "sum z within team for PTS REB AST STL BLK minus z for TOV",
    "teams": teams_out,
}
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)

print(str(out_path.resolve()))

/Users/pranav/Documents/GitHub/Software-Engineering/backend/player_rank_export.json
